<a href="https://colab.research.google.com/github/s-bishal/ML-Lab/blob/main/ML_Lab_Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [90]:
import pandas as pd
import numpy as np
import kagglehub
from kagglehub import KaggleDatasetAdapter

path="food_coded.csv"

df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS,"borapajo/food-choices",path)

df.head()

Using Colab cache for faster access to the 'food-choices' dataset.


,GPA,Gender,breakfast,calories_chicken,calories_day,calories_scone,coffee,comfort_food,comfort_food_reasons,comfort_food_reasons_coded,...,soup,sports,thai_food,tortilla_calories,turkey_calories,type_sports,veggies_day,vitamins,waffle_calories,weight
0,2.4,2,1,430,NaN,315.0,1,none,we dont have comfort,9.0,...,1.0,1.0,1,1165.0,345,car racing,5,1,1315,187
1,3.654,1,1,610,3.0,420.0,2,"chocolate, chips, ice cream","Stress, bored, anger",1.0,...,1.0,1.0,2,725.0,690,Basketball,4,2,900,155
2,3.3,1,1,720,4.0,420.0,2,"frozen yogurt, pizza, fast food","stress, sadness",1.0,...,1.0,2.0,5,1165.0,500,none,5,1,900,I'm not answering this.
3,3.2,1,1,430,3.0,420.0,2,"Pizza, Mac and cheese, ice cream",Boredom,2.0,...,1.0,2.0,5,725.0,690,NaN,3,1,1315,"Not sure, 240"
4,3.5,1,1,720,2.0,420.0,2,"Ice cream, chocolate, chips","Stress, boredom, cravings",1.0,...,1.0,1.0,4,940.0,500,Softball,4,2,760,190


### Data Preprocessing

First, let's get an overview of the dataset, including data types and non-null values. Then, we'll identify and handle missing values.

In [91]:
# Display basic info about the DataFrame to see data types and non-null counts
display(df.info())

# Check for missing values
missing_values = df.isnull().sum()

# Filter to show only columns with missing values
missing_values = missing_values[missing_values > 0]

print("\nColumns with Missing Values:")
display(missing_values.sort_values(ascending=False))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 61 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   GPA                           123 non-null    object 
 1   Gender                        125 non-null    int64  
 2   breakfast                     125 non-null    int64  
 3   calories_chicken              125 non-null    int64  
 4   calories_day                  106 non-null    float64
 5   calories_scone                124 non-null    float64
 6   coffee                        125 non-null    int64  
 7   comfort_food                  124 non-null    object 
 8   comfort_food_reasons          123 non-null    object 
 9   comfort_food_reasons_coded    106 non-null    float64
 10  cook                          122 non-null    float64
 11  comfort_food_reasons_coded.1  125 non-null    int64  
 12  cuisine                       108 non-null    float64
 13  diet_

None


Columns with Missing Values:


,0
type_sports,26
calories_day,19
comfort_food_reasons_coded,19
cuisine,17
exercise,13
employment,9
mother_education,3
father_profession,3
cook,3
eating_changes,3


### Data Cleaning: `GPA` and `weight` columns

The `GPA` and `weight` columns are currently of `object` type, but they represent numerical values. Some entries are non-numeric, which needs to be handled. I will convert these columns to numeric, coercing errors to `NaN`, and then fill any `NaN` values with the mean of the respective column.

In [92]:
# Convert 'GPA' and 'weight' to numeric, coercing errors to NaN
df['GPA'] = pd.to_numeric(df['GPA'], errors='coerce')
df['weight'] = pd.to_numeric(df['weight'], errors='coerce')

# Impute missing values in 'GPA' and 'weight' with their respective means
df['GPA'].fillna(df['GPA'].mean(), inplace=True)
df['weight'].fillna(df['weight'].mean(), inplace=True)

print("Data types after converting 'GPA' and 'weight':")
display(df[['GPA', 'weight']].info())

# Recheck for missing values to confirm imputation for these columns
print("\nMissing values after imputing 'GPA' and 'weight':")
display(df[['GPA', 'weight']].isnull().sum())

Data types after converting 'GPA' and 'weight':
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   GPA     125 non-null    float64
 1   weight  125 non-null    float64
dtypes: float64(2)
memory usage: 2.1 KB


/tmp/ipykernel_1120/2191904415.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['GPA'].fillna(df['GPA'].mean(), inplace=True)
/tmp/ipykernel_1120/2191904415.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try 

None


Missing values after imputing 'GPA' and 'weight':


,0
GPA,0
weight,0


### Drop Columns

Some columns are either text-based, contain redundant information, or have a very high number of missing values, making them less suitable for direct use in modeling without extensive natural language processing or specific domain knowledge. Following the example of `Basic_Data_Preprocessing.ipynb`, I'll drop these columns to simplify the dataset and focus on more directly usable features.

In [93]:
# Columns to drop: text descriptions, redundant identifiers, or too many missing values
columns_to_drop = [
    'comfort_food', 'comfort_food_reasons', 'type_sports', # Text descriptions/high missing
    'diet_current', 'eating_changes', 'father_profession', # Text descriptions
    'fav_cuisine', 'food_childhood', 'healthy_meal', 'ideal_diet', # Text descriptions
    'meals_dinner_friend', 'mother_profession' # Text descriptions
]

# Filter out columns that don't exist in the DataFrame
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]

df_processed = df.drop(columns=existing_columns_to_drop, errors='ignore')

print(f"Dropped columns: {existing_columns_to_drop}")
print(f"New DataFrame shape: {df_processed.shape}")

Dropped columns: ['comfort_food', 'comfort_food_reasons', 'type_sports', 'diet_current', 'eating_changes', 'father_profession', 'fav_cuisine', 'food_childhood', 'healthy_meal', 'ideal_diet', 'meals_dinner_friend', 'mother_profession']
New DataFrame shape: (125, 49)


### Separate Numerical and Categorical Features

Now that we've cleaned and dropped some columns, the next step is to clearly separate our features into numerical and categorical types. This distinction is crucial for applying appropriate preprocessing techniques later, such as imputation for numerical data and one-hot encoding for categorical data.

In [94]:
# Separate features into numerical and categorical
numerical_cols = df_processed.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df_processed.select_dtypes(include='object').columns.tolist()

print(f"Numerical columns ({len(numerical_cols)}):\n{numerical_cols}")
print(f"\nCategorical columns ({len(categorical_cols)}):\n{categorical_cols}")

Numerical columns (49):
['GPA', 'Gender', 'breakfast', 'calories_chicken', 'calories_day', 'calories_scone', 'coffee', 'comfort_food_reasons_coded', 'cook', 'comfort_food_reasons_coded.1', 'cuisine', 'diet_current_coded', 'drink', 'eating_changes_coded', 'eating_changes_coded1', 'eating_out', 'employment', 'ethnic_food', 'exercise', 'father_education', 'fav_cuisine_coded', 'fav_food', 'fries', 'fruit_day', 'grade_level', 'greek_food', 'healthy_feeling', 'ideal_diet_coded', 'income', 'indian_food', 'italian_food', 'life_rewarding', 'marital_status', 'mother_education', 'nutritional_check', 'on_off_campus', 'parents_cook', 'pay_meal_out', 'persian_food', 'self_perception_weight', 'soup', 'sports', 'thai_food', 'tortilla_calories', 'turkey_calories', 'veggies_day', 'vitamins', 'waffle_calories', 'weight']

Categorical columns (0):
[]


### Refine Feature Separation: Identifying Integer-Coded Categorical Features

Although `df_processed.select_dtypes(include=np.number)` identified many columns as numerical, some of these (e.g., 'Gender', 'breakfast', 'coffee') are actually categorical features represented by integers. These require one-hot encoding rather than numerical imputation or scaling. Let's explicitly separate these integer-coded categorical features from the truly continuous/numerical features.

In [98]:
# Based on the dataset description and common sense, identify integer-coded categorical columns
# that were mistakenly classified as numerical by df.select_dtypes(include=np.number)

# Assuming these are the columns that should be treated as categorical for one-hot encoding
categorical_cols_for_ohe = [
    'Gender', 'breakfast', 'coffee', 'comfort_food_reasons_coded.1',
    'diet_current_coded', 'eating_changes_coded', 'eating_changes_coded1',
    'eating_out', 'ethnic_food', 'fav_cuisine_coded', 'fries',
    'fruit_day', 'grade_level', 'greek_food', 'healthy_feeling',
    'ideal_diet_coded', 'indian_food', 'italian_food', 'nutritional_check',
    'parents_cook', 'pay_meal_out', 'thai_food', 'veggies_day', 'vitamins'
]

# Ensure these columns actually exist in df_processed
categorical_cols_for_ohe = [col for col in categorical_cols_for_ohe if col in df_processed.columns]

# Update numerical_cols by removing these identified categorical ones
numerical_cols = [col for col in numerical_cols if col not in categorical_cols_for_ohe]

print(f"Refined Numerical columns ({len(numerical_cols)}):\n{numerical_cols}")
print(f"\nCategorical columns for One-Hot Encoding ({len(categorical_cols_for_ohe)}):\n{categorical_cols_for_ohe}")

Refined Numerical columns (25):
['GPA', 'calories_chicken', 'calories_day', 'calories_scone', 'comfort_food_reasons_coded', 'cook', 'cuisine', 'drink', 'employment', 'exercise', 'father_education', 'fav_food', 'income', 'life_rewarding', 'marital_status', 'mother_education', 'on_off_campus', 'persian_food', 'self_perception_weight', 'soup', 'sports', 'tortilla_calories', 'turkey_calories', 'waffle_calories', 'weight']

Categorical columns for One-Hot Encoding (0):
[]


### Impute Missing Values in Numerical Columns

Now that we have a clear separation of numerical and categorical columns, we can proceed with imputing missing values in the numerical features. For simplicity and robustness, I'll use the mean strategy for imputation, similar to the `Basic_Data_Preprocessing.ipynb` example. This helps ensure that all numerical features are complete before further processing like scaling.

In [99]:
from sklearn.impute import SimpleImputer

# Impute missing values in numerical columns using the mean strategy
imputer = SimpleImputer(strategy='mean')

# Apply imputer to numerical columns in df_processed
df_processed[numerical_cols] = imputer.fit_transform(df_processed[numerical_cols])

print("Missing values in numerical columns after imputation:")
display(df_processed[numerical_cols].isnull().sum().sort_values(ascending=False))

Missing values in numerical columns after imputation:


,0
GPA,0
calories_chicken,0
calories_day,0
calories_scone,0
comfort_food_reasons_coded,0
cook,0
cuisine,0
drink,0
employment,0
exercise,0


### One-Hot Encode Categorical Features

With numerical features imputed, the next step is to convert the identified categorical features into a format suitable for machine learning algorithms using one-hot encoding. This will create new binary columns for each category, preventing the model from assuming any ordinal relationship where none exists.

In [100]:
# Apply one-hot encoding to the identified categorical columns
df_processed = pd.get_dummies(df_processed, columns=categorical_cols_for_ohe, drop_first=True, dtype=int)

print("DataFrame shape after one-hot encoding:")
print(df_processed.shape)
print("\nFirst 5 rows of the processed DataFrame with new one-hot encoded columns:")
display(df_processed.head())

DataFrame shape after one-hot encoding:
(125, 123)

First 5 rows of the processed DataFrame with new one-hot encoded columns:


,GPA,calories_chicken,calories_day,calories_scone,comfort_food_reasons_coded,cook,cuisine,drink,employment,exercise,...,pay_meal_out_6,thai_food_2,thai_food_3,thai_food_4,thai_food_5,veggies_day_2,veggies_day_3,veggies_day_4,veggies_day_5,vitamins_2
0,2.400,430.0,3.028302,315.0,9.0,2.0,1.388889,1.0,3.0,1.0,...,0,0,0,0,0,0,0,0,1,0
1,3.654,610.0,3.000000,420.0,1.0,3.0,1.000000,2.0,2.0,1.0,...,0,1,0,0,0,0,0,1,0,1
2,3.300,720.0,4.000000,420.0,1.0,1.0,3.000000,1.0,3.0,2.0,...,0,0,0,0,1,0,0,0,1,0
3,3.200,430.0,3.000000,420.0,2.0,2.0,2.000000,2.0,3.0,3.0,...,0,0,0,0,1,0,1,0,0,0
4,3.500,720.0,2.000000,420.0,1.0,1.0,2.000000,2.0,2.0,1.0,...,0,0,0,1,0,0,0,1,0,1


### Refine Feature Separation: Identifying Integer-Coded Categorical Features

Although `df_processed.select_dtypes(include=np.number)` identified many columns as numerical, some of these (e.g., 'Gender', 'breakfast', 'coffee') are actually categorical features represented by integers. These require one-hot encoding rather than numerical imputation or scaling. Let's explicitly separate these integer-coded categorical features from the truly continuous/numerical features.

In [95]:
# Based on the dataset description and common sense, identify integer-coded categorical columns
# that were mistakenly classified as numerical by df.select_dtypes(include=np.number)

# Assuming these are the columns that should be treated as categorical for one-hot encoding
categorical_cols_for_ohe = [
    'Gender', 'breakfast', 'coffee', 'comfort_food_reasons_coded.1',
    'diet_current_coded', 'eating_changes_coded', 'eating_changes_coded1',
    'eating_out', 'ethnic_food', 'fav_cuisine_coded', 'fries',
    'fruit_day', 'grade_level', 'greek_food', 'healthy_feeling',
    'ideal_diet_coded', 'indian_food', 'italian_food', 'nutritional_check',
    'parents_cook', 'pay_meal_out', 'thai_food', 'veggies_day', 'vitamins'
]

# Ensure these columns actually exist in df_processed
categorical_cols_for_ohe = [col for col in categorical_cols_for_ohe if col in df_processed.columns]

# Update numerical_cols by removing these identified categorical ones
numerical_cols = [col for col in numerical_cols if col not in categorical_cols_for_ohe]

print(f"Refined Numerical columns ({len(numerical_cols)}):\n{numerical_cols}")
print(f"\nCategorical columns for One-Hot Encoding ({len(categorical_cols_for_ohe)}):\n{categorical_cols_for_ohe}")

Refined Numerical columns (25):
['GPA', 'calories_chicken', 'calories_day', 'calories_scone', 'comfort_food_reasons_coded', 'cook', 'cuisine', 'drink', 'employment', 'exercise', 'father_education', 'fav_food', 'income', 'life_rewarding', 'marital_status', 'mother_education', 'on_off_campus', 'persian_food', 'self_perception_weight', 'soup', 'sports', 'tortilla_calories', 'turkey_calories', 'waffle_calories', 'weight']

Categorical columns for One-Hot Encoding (24):
['Gender', 'breakfast', 'coffee', 'comfort_food_reasons_coded.1', 'diet_current_coded', 'eating_changes_coded', 'eating_changes_coded1', 'eating_out', 'ethnic_food', 'fav_cuisine_coded', 'fries', 'fruit_day', 'grade_level', 'greek_food', 'healthy_feeling', 'ideal_diet_coded', 'indian_food', 'italian_food', 'nutritional_check', 'parents_cook', 'pay_meal_out', 'thai_food', 'veggies_day', 'vitamins']


### Impute Missing Values in Numerical Columns

Now that we have a clear separation of numerical and categorical columns, we can proceed with imputing missing values in the numerical features. For simplicity and robustness, I'll use the mean strategy for imputation, similar to the `Basic_Data_Preprocessing.ipynb` example. This helps ensure that all numerical features are complete before further processing like scaling.

In [96]:
from sklearn.impute import SimpleImputer

# Impute missing values in numerical columns using the mean strategy
imputer = SimpleImputer(strategy='mean')

# Apply imputer to numerical columns in df_processed
df_processed[numerical_cols] = imputer.fit_transform(df_processed[numerical_cols])

print("Missing values in numerical columns after imputation:")
display(df_processed[numerical_cols].isnull().sum().sort_values(ascending=False))

Missing values in numerical columns after imputation:


,0
GPA,0
calories_chicken,0
calories_day,0
calories_scone,0
comfort_food_reasons_coded,0
cook,0
cuisine,0
drink,0
employment,0
exercise,0


### One-Hot Encode Categorical Features

With numerical features imputed, the next step is to convert the identified categorical features into a format suitable for machine learning algorithms using one-hot encoding. This will create new binary columns for each category, preventing the model from assuming any ordinal relationship where none exists.

In [97]:
# Apply one-hot encoding to the identified categorical columns
df_processed = pd.get_dummies(df_processed, columns=categorical_cols_for_ohe, drop_first=True, dtype=int)

print("DataFrame shape after one-hot encoding:")
print(df_processed.shape)
print("\nFirst 5 rows of the processed DataFrame with new one-hot encoded columns:")
display(df_processed.head())

DataFrame shape after one-hot encoding:
(125, 123)

First 5 rows of the processed DataFrame with new one-hot encoded columns:


,GPA,calories_chicken,calories_day,calories_scone,comfort_food_reasons_coded,cook,cuisine,drink,employment,exercise,...,pay_meal_out_6,thai_food_2,thai_food_3,thai_food_4,thai_food_5,veggies_day_2,veggies_day_3,veggies_day_4,veggies_day_5,vitamins_2
0,2.400,430.0,3.028302,315.0,9.0,2.0,1.388889,1.0,3.0,1.0,...,0,0,0,0,0,0,0,0,1,0
1,3.654,610.0,3.000000,420.0,1.0,3.0,1.000000,2.0,2.0,1.0,...,0,1,0,0,0,0,0,1,0,1
2,3.300,720.0,4.000000,420.0,1.0,1.0,3.000000,1.0,3.0,2.0,...,0,0,0,0,1,0,0,0,1,0
3,3.200,430.0,3.000000,420.0,2.0,2.0,2.000000,2.0,3.0,3.0,...,0,0,0,0,1,0,1,0,0,0
4,3.500,720.0,2.000000,420.0,1.0,1.0,2.000000,2.0,2.0,1.0,...,0,0,0,1,0,0,0,1,0,1


### Scale Numerical Features

To ensure that all numerical features contribute equally to the model training, we need to scale them. I will use `StandardScaler` to transform the numerical features so they have a mean of 0 and a standard deviation of 1. This prevents features with larger values from dominating the learning algorithm. Since 'GPA' is our target variable, we will exclude it from scaling during this step.

In [101]:
from sklearn.preprocessing import StandardScaler

# Identify numerical columns to scale (exclude 'GPA' if it's the target variable)
numerical_features_to_scale = [col for col in numerical_cols if col != 'GPA']

# Initialize the StandardScaler
scaler = StandardScaler()

# Apply scaling to the numerical features in df_processed
df_processed[numerical_features_to_scale] = scaler.fit_transform(df_processed[numerical_features_to_scale])

print("Numerical features scaled successfully.")
print("First 5 rows of df_processed after scaling numerical features:")
display(df_processed[numerical_features_to_scale].head())

Numerical features scaled successfully.
First 5 rows of df_processed after scaling numerical features:


,calories_chicken,calories_day,calories_scone,comfort_food_reasons_coded,cook,cuisine,drink,employment,exercise,father_education,...,mother_education,on_off_campus,persian_food,self_perception_weight,soup,sports,tortilla_calories,turkey_calories,waffle_calories,weight
0,-1.127263,0.000000,-0.830800,3.486702,-0.770246,0.000000,-1.139541,1.078877,-0.940175,1.267079,...,-2.106432,-0.477296,1.553078,-0.109274,-0.529712,-0.806478,1.084565,-1.384030,0.975490,0.919749
1,0.250061,-0.048302,-0.372258,-0.939531,0.208608,-0.431212,0.891815,-0.876587,-0.940175,-1.240120,...,0.498143,-0.477296,0.845057,-0.109274,-0.529712,-0.806478,-1.110311,0.889301,-0.700124,-0.112952
2,1.091758,1.658370,-0.372258,-0.939531,-1.749100,1.786451,-1.139541,1.078877,0.655273,-1.240120,...,-1.238240,1.002321,1.553078,2.600715,-0.529712,1.260122,1.084565,-0.362679,-0.700124,0.000000
3,-1.127263,-0.048302,-0.372258,-0.386251,-0.770246,0.677619,0.891815,1.078877,2.250721,-1.240120,...,0.498143,-0.477296,1.553078,1.697385,-0.529712,1.260122,-1.110311,0.889301,0.975490,0.000000
4,1.091758,-1.754974,-0.372258,-0.939531,-1.749100,0.677619,0.891815,-0.876587,-0.940175,0.431346,...,1.366334,-0.477296,-0.570985,0.794056,-0.529712,-0.806478,-0.037815,-0.362679,-1.265391,1.016565


### Separate Features (X) and Target (y), then Split Data

With all preprocessing steps applied, the dataset is now ready for model training. I will separate the features (X) from the target variable (y), which is 'GPA'. Then, I will split the data into training and testing sets to evaluate how well our model generalizes to unseen data.

In [102]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = df_processed.drop('GPA', axis=1)
y = df_processed['GPA']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (100, 122)
X_test shape: (25, 122)
y_train shape: (100,)
y_test shape: (25,)
